---
layout: post
codemirror: true
title: Adventure Game — Custom Level Documentation
description: A walkthrough of three custom game levels built with the GameEngine framework, CS 113 requirement alignment, mechanic deep-dives, and a future development roadmap.
permalink: /space-blog
author: Team Bob
---

## About These Levels

Three custom levels were built using the **GameEngine framework** and exported from **GameBuilder**. Each level introduces a new mechanic — from basic NPC interaction to invisible maze navigation to a chase-and-survive AI challenge.

| # | Level | Core Mechanic |
|---|-------|---------------|
| 1 | **Space Level** | NPC dialogue + level transition via `continue = false` |
| 2 | **Alien Maze** | Invisible barriers, red glow on collision, timed run |
| 3 | **Alien Chase** | Chase AI, survival countdown, rage speed, E-key restart |

This blog documents each level's design alongside its alignment to the **CS 113 course requirements**. Requirements already demonstrated in the current codebase are marked ✅. Requirements planned for a future sprint are marked 🔲 and listed in the [Development Roadmap](#development-roadmap) at the bottom.

---

## CS 113 Requirements — At a Glance

The table below maps every CS 113 learning objective to its current status across the three levels.

### Object-Oriented Design

| Requirement | Status | Where |
|-------------|--------|-------|
| **Classes** — write classes, instantiate objects, call methods | ✅ | `GameLevelSpace`, `GameLevel2`, `GameLevelstuck_final` are all ES6 classes with constructors |
| **Encapsulation** — private data, public interface | ✅ | `AlienChaseAI` and `SurvivalManager` expose only `start/stop/pause/resume/caught/reset`; internal state (`offsetX`, `stillSince`, `countdownId`) is hidden |
| **Abstraction** — hide complexity behind methods | ✅ | `makeBarrier()` abstracts barrier creation; `_glowBarrier()`, `_showRestartFlash()`, `_showVictoryScreen()` abstract all DOM work |
| **Inheritance** — extend base classes | ✅ | `Player`, `Npc`, `Barrier` extend `GameEnv` base classes; all three level classes follow the engine's level contract |
| **Polymorphism** — override methods for flexible design | ✅ | Each NPC overrides `reaction()` and `interact()` differently per level; `onCollide` / `onBarrierCollision` are polymorphic collision hooks |

### Data Types & Structures

| Requirement | Status | Where |
|-------------|--------|-------|
| **Numbers** | ✅ | `SCALE_FACTOR`, `STEP_FACTOR`, `offsetX/Y`, `remaining`, coordinates throughout |
| **Strings** | ✅ | NPC `dialogues` arrays, `id` fields, DOM `cssText` strings, image `src` paths |
| **Booleans** | ✅ | `visible`, `paused`, `frozen`, `levelTransitionTriggered` flags |
| **Arrays** | ✅ | `this.classes`, `dialogues`, `sources` list in `_findPlayer()` |
| **JSON Objects** | ✅ | Every data blob (`playerData`, `npcData1`, `CHASE_CONFIG`) is a plain JS object |
| **Collections / Lists (Java backend)** | 🔲 | Sprint 1 — leaderboard `ArrayList<Score>` in Spring Boot |
| **Stacks / Queues** | 🔲 | Sprint 2 — undo-move stack in the maze using `Deque<Position>` |
| **Trees** | 🔲 | Sprint 3 — Decision Tree skill classifier via Smile/Weka |
| **Sets** | 🔲 | Sprint 1 — `HashSet<String>` for unique player usernames |
| **Dictionaries / Maps** | 🔲 | Sprint 1 — `HashMap<String, Integer>` for level high-scores |
| **Graphs** | 🔲 | Sprint 4 — leaderboard challenge graph with BFS traversal |

### Algorithms

| Requirement | Status | Where |
|-------------|--------|-------|
| **Searching** | ✅ | `_findPlayer()` in `GameLevel2` — O(n) linear search across game object stores |
| **Sorting** | 🔲 | Sprint 2 — leaderboard sorted by time via `Comparator` |
| **Hashing** | 🔲 | Sprint 1 — BCrypt password hashing in Spring Security |
| **Algorithm Analysis (Big-O)** | 🔲 | Sprint 2 — blog section with O(n), O(n log n), O(1) annotations |

### Control Structures

| Requirement | Status | Where |
|-------------|--------|-------|
| **Iteration** | ✅ | `setInterval` chase loop and countdown loop; `for...of` in `_findPlayer()` |
| **Conditions** | ✅ | Cooldown guards, distance checks, stillness checks |
| **Nested conditions** | ✅ | Rage speed logic inside stillness timer inside movement check in `_tick()` |
| **try/catch** | 🔲 | Sprint 1 — error handling around DOM operations and Java service calls |
| **.then/.catch (Promises)** | 🔲 | Sprint 2 — async leaderboard fetch |

### Input / Output

| Requirement | Status | Where |
|-------------|--------|-------|
| **HTML5 Input** | ✅ | Keyboard input via `keypress` map (WASD / Arrow keys) |
| **DOM Manipulation** | ✅ | Overlays, HUD, glow divs injected and removed via `document.createElement` |
| **Input Validation** | 🔲 | Sprint 2 — leaderboard score submission form with JS validation |

### Software Development

| Requirement | Status | Where |
|-------------|--------|-------|
| **Version Control** | ✅ | Levels committed to repo; imported by path |
| **Code Comments (JSDoc)** | ✅ | `@description`, `@method`, `@param`, `@property` blocks throughout Level 3 |
| **Help Documentation** | ✅ | This blog + Level 2 startup popup + Level 3 CAUGHT overlay |
| **API Development** | 🔲 | Sprint 3 — RESTful leaderboard endpoints |
| **Database Integration** | 🔲 | Sprint 3 — JPA/Hibernate `Score` entity with relationships |
| **Testing** | 🔲 | Sprint 2 — JUnit tests for `ScoreService` and `PlayerRepository` |
| **Docker / DNS / nginx** | 🔲 | Sprint 4 — full deployment stack |
| **CI/CD** | 🔲 | Sprint 4 — GitHub Actions on merge to `main` |

## Quick Controls Reference

| Level | Move | Interact |
|-------|------|----------|
| Space Level | ← ↑ → ↓ Arrow Keys | Walk into Chill Guy NPC |
| Alien Maze | W A S D | Walk to R2, press **E** |
| Alien Chase | W A S D | Press **E** near alien to restart after being caught |

---

## Level 1: Space Level

> An introductory level set on an alien planet. Navigate past a set of visible barriers to reach Chill Guy, who will send you to the next level.

### How It Works

**NPC Interaction and Level Transition**

The `Npc` object (`npcData1`) overrides two callbacks that the engine calls at specific moments:

- `reaction()` — fires every frame the player's bounding box overlaps the NPC hitbox. Used here to display a dialogue line.
- `interact()` — fires when the player presses **E** within range. The key line is:

```javascript
interact: function() {
    if (this.dialogueSystem) { this.showRandomDialogue(); }
    // Trigger level transition — same effect as pressing ESC
    if (this.gameEnv && this.gameEnv.gameControl &&
            this.gameEnv.gameControl.currentLevel) {
        this.gameEnv.gameControl.currentLevel.continue = false;
    }
}
```

Setting `currentLevel.continue = false` signals the `GameControl` loop to tear down the current level and advance to the next one in `gameLevelClasses`.

**Barrier Layout**

Five `Barrier` objects are placed using absolute pixel coordinates (`x`, `y`, `width`, `height`). They are marked `visible: true` so the player can see and navigate around them. The `fromOverlay: true` flag marks them as builder-managed.

```javascript
const dbarrier_1 = {
    id: 'dbarrier_1', x: 700, y: 100, width: 150, height: 20,
    visible: true,
    hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
    fromOverlay: true
};
```

### CS 113 Requirements Demonstrated — Level 1

**✅ Classes & Objects**

The entire level is a single ES6 class `GameLevelSpace`. The constructor receives a `gameEnv` object, builds data objects for each game entity, and registers them in `this.classes`. This directly demonstrates *writing classes*, *creating objects*, and *instantiating* them via the `{ class: Player, data: playerData }` pattern.

**✅ Polymorphism — Overriding `reaction()` and `interact()`**

The base `Npc` class defines `reaction()` and `interact()` as empty functions. In Level 1, both are overridden with custom behaviour specific to this NPC. This is runtime polymorphism: the engine calls the same method name on every NPC, but each executes its own version.

```javascript
// Base Npc class defines these as empty functions.
// Level 1 overrides them with level-specific logic:
reaction: function() { if (this.dialogueSystem) { this.showReactionDialogue(); } },
interact: function() {
    if (this.dialogueSystem) { this.showRandomDialogue(); }
    this.gameEnv.gameControl.currentLevel.continue = false; // advance level
}
```

**✅ Arrays**

`this.classes` is an array of objects — each entry specifies a class constructor and a data blob. The engine iterates this array to instantiate every game object in the level. The `dialogues` property on the NPC is also an array rotated on each interaction.

**✅ JSON Objects**

Every game entity (`playerData`, `npcData1`, `dbarrier_1`) is configured as a plain JS object with typed properties — numbers, strings, booleans, and nested objects — directly mirroring a JSON document structure.

**✅ HTML5 Input**

The player's movement is driven by `keypress: { up: 38, left: 37, down: 40, right: 39 }` — arrow key keycodes. The engine reads these each frame and translates them into player movement on the canvas.

In [ ]:
%%js

// GAME_RUNNER: Navigate the alien planet and interact with Chill Guy to advance to the next level. Use Arrow Keys to move.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevelSpacelevel3 from '/assets/js/adventureGame/GameLevelSpacelevel3.js';
export const gameLevelClasses = [GameLevelSpacelevel3];
export { GameControl };

<IPython.core.display.Javascript object>

---

## Level 2: Alien Maze

> The walls are invisible. Touch one and you restart from the beginning. Navigate to R2 to complete the level — your time is being tracked!

### How It Works

**Invisible Barriers with Red Glow on Collision**

All barriers are created with `visible: false`. When a collision is detected, the `onCollide` callback injects a short-lived `<div>` over the barrier's canvas coordinates and resets the player to `INIT_POSITION`:

```javascript
const makeBarrier = (id, x, y, w, h) => ({
    id, x, y, width: w, height: h,
    visible: false,
    onCollide: function () {
        _glowBarrier(this);          // flash red overlay at barrier position
        const player = GameLevel2._findPlayer(gameEnv);
        if (player) {
            player.x = player.data.INIT_POSITION.x;   // reset to spawn
            player.y = player.data.INIT_POSITION.y;
        }
        GameLevel2._showRestartFlash();
    }
});
```

**Startup Popup and Victory Screen**

`_showStartupPopup()` appends a briefing overlay to `document.body`. When the player clicks **START MISSION**, `GameLevel2._startTime = Date.now()` begins the run timer. When R2 is reached, elapsed time is computed as:

```javascript
const elapsed = Math.floor((Date.now() - GameLevel2._startTime) / 1000);
```

**Maze Layout (fractional coordinates)**

Barrier positions use fractions (0.0–1.0) scaled to the viewport at runtime. The left wall has an entrance gap between `y: 0.35` and `y: 0.55`:

```javascript
const mazeLeftTop    = makeBarrier('maze_left_top',    0.20, 0.15, 0.02, 0.20);
const mazeLeftBottom = makeBarrier('maze_left_bottom', 0.20, 0.55, 0.02, 0.30);
// gap between y=0.35 and y=0.55 is the only entrance
```

### CS 113 Requirements Demonstrated — Level 2

**✅ Abstraction — `makeBarrier()` Factory Function**

Rather than duplicating nine nearly identical barrier objects, Level 2 defines a `makeBarrier(id, x, y, w, h)` factory. All collision logic, glow behaviour, and player reset are encapsulated inside it. Callers only provide coordinates. This is *abstraction*: hiding complex implementation behind a simple, reusable interface.

```javascript
// Caller — clean and readable:
const mazeWall1 = makeBarrier('maze_wall_1', 0.30, 0.25, 0.02, 0.30);
// All collision logic lives inside makeBarrier — never repeated.
```

**✅ Searching — `_findPlayer()` Linear Search**

The static method `GameLevel2._findPlayer(gameEnv)` performs a linear search across multiple candidate object arrays (`gameEnv.gameObjects`, `gameEnv.objects`, `gameEnv.gameControl.gameObjects`). It iterates each list and returns the first object whose `id` matches `'playerData'`. This is CS 113's *searching* requirement in action.

```javascript
// O(n) linear search across all game objects:
const found = arr.find(o => o?.data?.id === 'playerData' || o?.id === 'playerData');
```

*Big-O: O(n) where n = total game objects. Acceptable here because n < 20 per level.*

**✅ DOM Input/Output — Overlays and Glow Effect**

Three DOM helper methods (`_showStartupPopup`, `_showRestartFlash`, `_showVictoryScreen`) programmatically create, style, and remove HTML elements at runtime. This demonstrates *HTML5 Input/Output*: reading user button clicks, building dynamic UI elements, and manipulating the Document Object Model — all core CS 113 I/O requirements.

**✅ Conditions and Iteration**

`_findPlayer()` uses a `for...of` loop (iteration) with optional chaining guards (`o?.data?.id`) for safe nested property access (condition). The `onCollide` handler uses a null-check condition before resetting the player.

**✅ Static Class Properties**

`GameLevel2._startTime` is a static property — set once when the player starts, read when the victory screen computes elapsed time. This demonstrates the distinction between class-level (static) state and instance-level state, an important OOP concept.

In [ ]:
%%js

// GAME_RUNNER: Find R2 hidden in the invisible maze — touch a wall and you restart! Use WASD to move.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevel2 from '/assets/js/adventureGame/GameLevel2.js';
export const gameLevelClasses = [GameLevel2];
export { GameControl };

<IPython.core.display.Javascript object>

### Level 2 — Mini-Game (No Code Editor)

> Play the maze without the code editor — good for embedding in lesson articles.

In [ ]:
%%js

// GAME_RUNNER: Can you beat the invisible maze? Find R2 to win! | hide_edit: true

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevel2 from '/assets/js/adventureGame/GameLevel2.js';
export const gameLevelClasses = [GameLevel2];
export { GameControl };

---

## Level 3: Alien Chase — Survive!

> An alien slime is hunting you. Survive for 10 seconds to win. Stand still too long and it enters **rage mode**. If caught, press **E** near the alien to restart.

### How It Works

This level runs two independent systems alongside the standard game loop.

**`AlienChaseAI` — The Chase Loop**

`AlienChaseAI` runs its own `setInterval` at `TICK_MS: 33` (≈ 30 fps). Each tick it reads canvas positions via `getBoundingClientRect()`, computes a normalised direction vector, and applies a CSS transform:

```javascript
npcCanvas.style.transform = `translate(${this.offsetX}px, ${this.offsetY}px)`;
```

**Stillness Detection → Rage Mode**

Each tick, the AI measures px moved since the last tick. Once the player has been still for `STILL_THRESHOLD_MS` (3 s), the alien switches to `RAGE_SPEED: 3.8` px/tick and the HUD turns red:

```javascript
if ((now - this.stillSince) >= CHASE_CONFIG.STILL_THRESHOLD_MS) {
    this.currentSpeed = CHASE_CONFIG.RAGE_SPEED;
    this._setRageVisual(true);
}
```

**Full Interaction Chain**

```
Player enters Alien hitbox
  └─ NPC.reaction() → SurvivalManager.caught()
       ├─ frozen = true  |  countdown stopped
       ├─ CAUGHT overlay shown
       └─ AlienChaseAI.pause()

Player presses E
  └─ NPC.interact() → SurvivalManager.reset()
       ├─ remaining = 10000ms  |  frozen = false
       ├─ Overlays hidden
       └─ AlienChaseAI.resume()  (alien teleports to spawn)

Countdown reaches zero
  └─ SurvivalManager.showWin()
       ├─ WIN overlay shown  |  AlienChaseAI.pause()
       └─ (after 4 s) currentLevel.continue = false → next level
```

### CS 113 Requirements Demonstrated — Level 3

**✅ Encapsulation — `AlienChaseAI` and `SurvivalManager`**

Both are namespace objects with clearly separated public APIs and hidden internal state. External code only ever calls `start()`, `stop()`, `pause()`, `resume()`, `caught()`, `reset()`. Internal properties (`offsetX`, `offsetY`, `stillSince`, `countdownId`, `lastResetTime`) are never accessed from outside — they are fully encapsulated.

**✅ Abstraction — `CHASE_CONFIG` and `INTERACTION_CONFIG`**

All AI tuning values are centralised in two config objects at the top of the file. This abstracts the *what* from the *how*. To rebalance the game feel, you edit constants — not the algorithm. This is the same principle as an abstract interface in Java: define the contract at the top, implement details below.

```javascript
// All game-feel tuning in one place:
const CHASE_CONFIG = {
    BASE_SPEED: 1.2, RAGE_SPEED: 3.8,
    STILL_THRESHOLD_MS: 3000, STILL_RADIUS: 4
};
```

**✅ Iteration — Concurrent `setInterval` Loops**

Two concurrent `setInterval` loops run simultaneously: `AlienChaseAI` ticks every 33ms; `SurvivalManager` ticks every 100ms. Both loops are started, paused, and cleared independently — demonstrating controlled iteration with state management, a direct CS 113 control structures requirement.

**✅ Nested Conditions — Stillness + Rage Logic**

Inside `_tick()`, conditions are nested three levels deep: *(1) is the AI paused?* → *(2) did the player move?* → *(3) has the stillness timer exceeded the threshold?*. This is exactly the *nested conditions* requirement from CS 113 control structures.

**✅ Operators — Mathematical and Boolean**

The direction vector uses mathematical operators; the cooldown guard uses a compound boolean expression:

```javascript
// Mathematical:
const dist  = Math.sqrt(dx * dx + dy * dy);
const normX = dx / dist;
// Boolean compound:
if (this.lastResetTime !== null &&
    Date.now() - this.lastResetTime < INTERACTION_CONFIG.CATCH_COOLDOWN_MS) return;
```

**✅ Code Comments — JSDoc**

Level 3 is the most thoroughly documented file. Every namespace, method, config constant, and NPC object has a JSDoc block with `@description`, `@method`, `@param`, `@property`, and `@type` tags. This directly satisfies the CS 113 *Code Comments* requirement: >10% comment density and JavaDoc-style documentation on all public methods.

**✅ Help System — In-Game Overlays as User Documentation**

The CAUGHT overlay (`☠ CAUGHT! Press E near the Alien to restart`) is live, in-game help. It tells the player exactly what to do next without leaving the game. This fulfils the CS 113 *Help System / user guide* requirement: contextual documentation embedded in the product.

In [ ]:
%%js

// GAME_RUNNER: Survive 10 seconds without being caught by the alien slime! Stand still and it goes into rage mode. Use WASD to move, E near the alien to restart.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevelstuck_final from '/assets/js/adventureGame/GameLevelstuck_final.js';
export const gameLevelClasses = [GameLevelstuck_final];
export { GameControl };

<IPython.core.display.Javascript object>

### Level 3 — Mini-Game (No Code Editor)

In [ ]:
%%js

// GAME_RUNNER: Survive the alien chase! Can you last 10 seconds? | hide_edit: true

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevelstuck_final from '/assets/js/adventureGame/GameLevelstuck_final.js';
export const gameLevelClasses = [GameLevelstuck_final];
export { GameControl };

---

## Playing All Three Levels in Sequence

Pass all three classes to `gameLevelClasses` and the engine advances automatically as each level's `continue` flag is set to `false`.

In [ ]:
%%js

// GAME_RUNNER: Play all three levels back to back — Space Level, Alien Maze, then Alien Chase. Use Arrow Keys for Level 1, WASD for Levels 2 and 3.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevelSpacelevel3 from '/assets/js/adventureGame/GameLevelSpacelevel3.js';
import GameLevel2 from '/assets/js/adventureGame/GameLevel2.js';
import GameLevelstuck_final from '/assets/js/adventureGame/GameLevelstuck_final.js';
export const gameLevelClasses = [GameLevelSpacelevel3, GameLevel2, GameLevelstuck_final];
export { GameControl };

<IPython.core.display.Javascript object>

---

## Development Roadmap

The requirements below are not yet implemented in the current three levels. Each is assigned to a future sprint with a concrete plan for where and how it will be added.

---

### 🔲 Sprint 1 — Backend Foundation & Auth
*Target: ~2 weeks*

| Requirement | Implementation Plan |
|-------------|---------------------|
| **Collections / Lists** | Build a Spring Boot backend with a `ScoreService` using `ArrayList<Score>` to store leaderboard entries. Demonstrate `add()`, `remove()`, and iteration. |
| **Sets** | Use `HashSet<String>` to track unique player usernames — prevents duplicates automatically without manual checks. |
| **Dictionaries / Maps** | Use `HashMap<String, Integer>` for in-memory level high-score lookup: key = level name, value = best time in seconds. |
| **Hashing** | Implement BCrypt password hashing in Spring Security for player login. Store only hashed passwords; compare via `BCrypt.checkpw()` — never plain-text. |
| **try/catch** | Wrap all DOM overlay operations in JS `try/catch`; wrap all database calls in Java `try/catch` blocks in the service layer. |

---

### 🔲 Sprint 2 — Algorithms, Sorting & Testing
*Target: ~4 weeks*

| Requirement | Implementation Plan |
|-------------|---------------------|
| **Sorting** | Add a leaderboard endpoint that sorts `List<Score>` by completion time using `Comparator.comparingInt(Score::getTime)`. Add a secondary sort by player name for ties. |
| **Algorithm Analysis (Big-O)** | Document in this blog: `_findPlayer()` = O(n), leaderboard sort = O(n log n), `HashMap` lookup = O(1). Add complexity comments to all key methods. |
| **Stacks / Queues** | Add an *undo last move* feature to the Alien Maze using a `Deque<Position>` as a stack — each step pushes a `{x, y}` snapshot; pressing U pops and restores it. |
| **Input Validation** | Add a score submission form (player name + time) with JS validation: non-empty name, time > 0, length limit. Show inline error messages before POST. |
| **Testing** | Write JUnit tests for `ScoreService` sort logic and `PlayerRepository` query results. Target >50% coverage on the service layer. |
| **.then/.catch (Promises)** | Refactor the leaderboard fetch to: `fetch('/api/scores').then(res => res.json()).then(renderBoard).catch(err => showError(err))`. |

---

### 🔲 Sprint 3 — Database, Trees & Async Assets
*Target: ~6 weeks*

| Requirement | Implementation Plan |
|-------------|---------------------|
| **Database Integration** | Add JPA/Hibernate `Score` entity with `@ManyToOne` relationship to `Player`. Persist run times to SQLite (dev) / MySQL (prod). Implement `ScoreRepository extends JpaRepository`. |
| **API Development** | Build RESTful endpoints: `GET /api/scores` (leaderboard), `POST /api/scores` (submit run), `DELETE /api/scores/{id}` (admin reset). Use `ResponseEntity` with proper HTTP status codes (200, 201, 404, 400). |
| **Trees** | Implement a Decision Tree to classify player skill level (beginner / intermediate / expert) based on maze completion time and wall-hit count. Integrate the Smile ML library. |
| **Async Asset Loading** | Use `Promise.all([...assets.map(src => loadImage(src))])` to preload all sprites before the level canvas appears. `.then` → start game, `.catch` → show a friendly error screen. |

---

### 🔲 Sprint 4 — Deployment & Graphs
*Target: ~8 weeks*

| Requirement | Implementation Plan |
|-------------|---------------------|
| **Graphs** | Model the leaderboard as a directed challenge graph: each player is a node, an edge means "player A beat player B's time". Implement BFS from a new player to find the shortest improvement path to the top score. |
| **Docker** | Write a `Dockerfile` for the Spring Boot backend and a `docker-compose.yml` combining Jekyll frontend + Java backend + MySQL. |
| **DNS Configuration** | Point a custom subdomain (`game.myproject.com`) to the deployment server with an A record. Verify with `dig` and `nslookup`. |
| **nginx** | Configure nginx as a reverse proxy: serve Jekyll static files on port 80, forward `/api/*` to the Spring Boot container on port 8080. |
| **CI/CD** | Add a GitHub Actions workflow: run JUnit tests on every PR; auto-deploy to the server on merge to `main`. |

---

## Image Asset Reference

All levels resolve assets via `gameEnv.path`. Ensure the following files exist before publishing:

| Asset | Path |
|-------|------|
| Alien planet background | `images/gamebuilder/bg/alien_planet.jpg` |
| Astronaut player sprite | `images/gamebuilder/sprites/astro.png` |
| Chill Guy NPC (Level 1) | `images/gamify/chillguy.png` |
| R2 robot NPC (Level 2) | `images/gamify/r2_idle.png` |
| Alien slime NPC (Level 3) | `images/gamebuilder/sprites/slime.png` |

---

## Build and Test

After editing this notebook, run `make` to convert it to an interactive HTML page:

```bash
make
# Then open: http://localhost:4000/adventure-game-levels
```

**Checklist before publishing:**

- [ ] Each game-runner cell renders a canvas with Play / Stop / Reset controls.
- [ ] Level transitions work (NPC interact or ESC key advances to the next level).
- [ ] Level 2 startup popup and victory screen appear and close correctly.
- [ ] Level 3 HUD timer counts down; CAUGHT and WIN overlays display correctly.
- [ ] `hide_edit: true` cells hide the code editor and show only the game canvas.
- [ ] All five image assets resolve without 404 errors (check browser console).